# Ledger Lens — Data Preprocessing & Quality Assurance Pipeline
## Problem Statement: SIH26102 (MPLADS AI Risk Intelligence & Monitoring)
**Phase 1: Standardization, Data Quality Flagging, and Clean Dataset Generation**

### Objectives
1. Load raw datasets safely and immutably from `data/raw/`.
2. Standardize column names into uniform snake_case with explicit documentation.
3. Clean Indian Rupee currency strings and handle formatting without loss of numerical precision.
4. Detect and partition summary trailer records (`Grand Total`) to maintain statistical integrity.
5. Identify and document missing values and domain duplicates without ungrounded synthetic imputation or data deletion.
6. Generate domain-specific data quality flags (`is_missing_amount`, `is_invalid_amount`, `is_duplicate_constituency`, `is_data_quality_issue`).
7. Persist validated datasets to `data/processed/` in CSV, Parquet, and JSON formats.


In [1]:
import os
import sys
import re
import json
import pandas as pd
import numpy as np

# Ensure UTF-8 output handling
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

# Add backend directory to path to enable reusable modules
module_path = os.path.abspath(os.path.join("..", "backend"))
if module_path not in sys.path:
    sys.path.append(module_path)

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Environment and paths initialized successfully.")


Environment and paths initialized successfully.


## 1. Raw Dataset Ingestion
Load `Allocated Limit for Honble MPs.csv` directly from `data/raw/` preserving raw immutability.


In [2]:
raw_dir = os.path.join("..", "data", "raw")
if not os.path.exists(raw_dir):
    raw_dir = os.path.join("data", "raw")

raw_filepath = os.path.join(raw_dir, "Allocated Limit for Honble MPs.csv")
raw_df = pd.read_csv(raw_filepath, dtype=str)

print(f"Loaded Raw Dataset: {raw_filepath}")
print(f"Raw Dimensions: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns")
print(f"Raw Columns: {list(raw_df.columns)}")


Loaded Raw Dataset: data\raw\Allocated Limit for Honble MPs.csv
Raw Dimensions: 544 rows × 5 columns
Raw Columns: ['Sr. No.', 'State', "Hon'ble Members of Parliaments", 'Constituency', 'Allocated AMOUNT ( ₹ )']


## 2. Column Standardization & Rename Documentation
Standardize column names to uniform lowercase `snake_case` format and document the mapping.


In [3]:
COLUMN_RENAME_MAP = {
    "Sr. No.": "sr_no",
    "State": "state",
    "Hon'ble Members of Parliaments": "mp_name",
    "Constituency": "constituency",
    "Allocated AMOUNT ( ₹ )": "allocated_amount_inr"
}

rename_doc = pd.DataFrame([
    {"Original Column": k, "Standardized Column": v, "Data Type Target": "int / str" if k == "Sr. No." else "float64" if "AMOUNT" in k else "str"}
    for k, v in COLUMN_RENAME_MAP.items()
])
print("Column Renaming & Standardization Specification:")
print(rename_doc.to_string(index=False))

df_standardized = raw_df.rename(columns=COLUMN_RENAME_MAP).copy()
print("\nRenamed DataFrame Head:")
print(df_standardized.head(3).to_string())


Column Renaming & Standardization Specification:
               Original Column  Standardized Column Data Type Target
                       Sr. No.                sr_no        int / str
                         State                state              str
Hon'ble Members of Parliaments              mp_name              str
                  Constituency         constituency              str
        Allocated AMOUNT ( ₹ ) allocated_amount_inr          float64

Renamed DataFrame Head:
  sr_no              state                         mp_name   constituency allocated_amount_inr
0     1        Maharashtra  AASHTIKAR PATIL NAGESH BAPURAO        HINGOLI            190289442
1     2  Jammu And Kashmir             ABDUL RASHID SHEIKH     BARAMULLAH         154773472.11
2     3              Bihar               ABHAY KUMAR SINHA  AURANGABAD_BR            147000000


## 3. Summary Trailer Record Partitioning
The official CSV includes a summary record at row index 543 (`sr_no = 'Grand Total'`).
To preserve data integrity, we:
1. Extract the official Grand Total value (`₹83,33,99,05,622.01`) into audit metadata.
2. Partition the 543 operational MP records into the primary data table.
3. Verify that zero operational records are lost.


In [4]:
is_grand_total = df_standardized['sr_no'].astype(str).str.strip().str.lower() == 'grand total'
trailer_df = df_standardized[is_grand_total].copy()
operational_df = df_standardized[~is_grand_total].copy()

print(f"Summary Trailer Records Isolated: {len(trailer_df)}")
print(trailer_df.to_string())

print(f"\nOperational MP Records Count: {len(operational_df)}")


Summary Trailer Records Isolated: 1
           sr_no state mp_name constituency allocated_amount_inr
543  Grand Total                              83,33,99,05,622.01

Operational MP Records Count: 543


## 4. Currency Parsing & Data Type Normalization
Clean the `allocated_amount_inr` field by stripping the Unicode `₹` symbol, commas, and whitespace, converting values to `float64`.


In [5]:
def parse_currency(val):
    if val is None or pd.isna(val):
        return 0.0, True, False
    val_str = str(val).strip()
    if val_str == "" or val_str.lower() in ("nan", "none", "null", "-"):
        return 0.0, True, False
    clean_str = re.sub(r"[^\d.]", "", val_str)
    try:
        amt = float(clean_str)
        if amt < 0:
            return amt, False, True
        return round(amt, 2), False, False
    except (ValueError, TypeError):
        return 0.0, False, True

# Parse trailer amount
gt_raw_val = trailer_df['allocated_amount_inr'].values[0]
official_grand_total, _, _ = parse_currency(gt_raw_val)

# Parse operational amounts
parsed_amounts = []
missing_flags = []
invalid_flags = []

for val in operational_df['allocated_amount_inr']:
    amt, is_miss, is_inv = parse_currency(val)
    parsed_amounts.append(amt)
    missing_flags.append(is_miss)
    invalid_flags.append(is_inv)

operational_df['allocated_amount_inr'] = parsed_amounts
operational_df['is_missing_amount'] = missing_flags
operational_df['is_invalid_amount'] = invalid_flags

# Normalize text and serial numbers
operational_df['sr_no'] = pd.to_numeric(operational_df['sr_no'], errors='coerce').fillna(0).astype(int)
operational_df['state'] = operational_df['state'].astype(str).str.strip()
operational_df['mp_name'] = operational_df['mp_name'].astype(str).str.strip()
operational_df['constituency'] = operational_df['constituency'].astype(str).str.strip()

print(f"Official Grand Total in Trailer: ₹{official_grand_total:,.2f}")
print(f"Calculated Sum of Operational MP Allocations: ₹{operational_df['allocated_amount_inr'].sum():,.2f}")
print(f"Reconciliation Delta: ₹{operational_df['allocated_amount_inr'].sum() - official_grand_total:,.2f}")


Official Grand Total in Trailer: ₹83,339,905,622.01
Calculated Sum of Operational MP Allocations: ₹83,339,905,622.01
Reconciliation Delta: ₹0.00


## 5. Reservation Category & Metadata Extraction
Extract parliamentary constituency reservation status (`SC` / `ST`) into explicit boolean features, and produce a normalized constituency name without tags.


In [6]:
is_sc = []
is_st = []
constituency_clean = []

for name in operational_df['constituency']:
    sc_match = bool(re.search(r"\(SC\)", name, re.IGNORECASE))
    st_match = bool(re.search(r"\(ST\)", name, re.IGNORECASE))
    
    clean_name = re.sub(r"\((SC|ST)\)", "", name, flags=re.IGNORECASE)
    clean_name = re.sub(r"_[A-Z]{2}$", "", clean_name)
    clean_name = re.sub(r"\s+", " ", clean_name).strip()
    
    is_sc.append(sc_match)
    is_st.append(st_match)
    constituency_clean.append(clean_name)

operational_df['is_reserved_sc'] = is_sc
operational_df['is_reserved_st'] = is_st
operational_df['constituency_clean'] = constituency_clean

print(f"SC Reserved Constituencies Count: {sum(is_sc)}")
print(f"ST Reserved Constituencies Count: {sum(is_st)}")
print(f"General / Unreserved Constituencies: {len(operational_df) - sum(is_sc) - sum(is_st)}")


SC Reserved Constituencies Count: 82
ST Reserved Constituencies Count: 44
General / Unreserved Constituencies: 417


## 6. Missing Value & Duplicate Analysis
Audit missing values and analyze duplicate constituencies grounded in real data.


In [7]:
# Missing Amount Audit
missing_records = operational_df[operational_df['is_missing_amount']]
print("=== Records with Missing Allocation Amount ===")
if not missing_records.empty:
    print(missing_records[['sr_no', 'state', 'mp_name', 'constituency', 'allocated_amount_inr']].to_string(index=False))
    print("\nGround Truth Explanation: Shri Chavan Vasantrao Balwantrao passed away in August 2024; seat transitioned via by-election.")
else:
    print("Zero missing amount records.")

# Duplicate Constituencies Audit
const_counts = operational_df['constituency'].value_counts()
duplicate_consts = const_counts[const_counts > 1].index.tolist()
operational_df['is_duplicate_constituency'] = operational_df['constituency'].isin(duplicate_consts)

print(f"\n=== Constituencies Appearing > 1 Time ({len(duplicate_consts)}) ===")
dup_df = operational_df[operational_df['is_duplicate_constituency']]
print(dup_df[['sr_no', 'state', 'mp_name', 'constituency', 'allocated_amount_inr', 'is_missing_amount']].to_string(index=False))


=== Records with Missing Allocation Amount ===
 sr_no       state                     mp_name constituency  allocated_amount_inr
   108 Maharashtra CHAVAN VASANTRAO BALWANTRAO       NANDED                  0.00

Ground Truth Explanation: Shri Chavan Vasantrao Balwantrao passed away in August 2024; seat transitioned via by-election.

=== Constituencies Appearing > 1 Time (1) ===
 sr_no       state                     mp_name constituency  allocated_amount_inr  is_missing_amount
   108 Maharashtra CHAVAN VASANTRAO BALWANTRAO       NANDED                  0.00               True
   390 Maharashtra   Ravindra Vasantrao Chavan       NANDED          147000000.00              False


## 7. Data Quality Flags & Composite Assessment
Create the composite quality flag `is_data_quality_issue` capturing all non-standard records requiring officer attention.


In [8]:
operational_df['is_data_quality_issue'] = (
    operational_df['is_missing_amount'] |
    operational_df['is_invalid_amount'] |
    operational_df['is_duplicate_constituency']
)

flag_summary = pd.DataFrame({
    'Flag Name': [
        'is_missing_amount',
        'is_invalid_amount',
        'is_duplicate_constituency',
        'is_reserved_sc',
        'is_reserved_st',
        'is_data_quality_issue'
    ],
    'Active Count': [
        operational_df['is_missing_amount'].sum(),
        operational_df['is_invalid_amount'].sum(),
        operational_df['is_duplicate_constituency'].sum(),
        operational_df['is_reserved_sc'].sum(),
        operational_df['is_reserved_st'].sum(),
        operational_df['is_data_quality_issue'].sum()
    ],
    'Percentage (%)': [
        (operational_df['is_missing_amount'].sum() / len(operational_df)) * 100,
        (operational_df['is_invalid_amount'].sum() / len(operational_df)) * 100,
        (operational_df['is_duplicate_constituency'].sum() / len(operational_df)) * 100,
        (operational_df['is_reserved_sc'].sum() / len(operational_df)) * 100,
        (operational_df['is_reserved_st'].sum() / len(operational_df)) * 100,
        (operational_df['is_data_quality_issue'].sum() / len(operational_df)) * 100
    ]
})

print("=== Data Quality Flags Summary ===")
print(flag_summary.to_string(index=False))


=== Data Quality Flags Summary ===
                Flag Name  Active Count  Percentage (%)
        is_missing_amount             1            0.18
        is_invalid_amount             0            0.00
is_duplicate_constituency             2            0.37
           is_reserved_sc            82           15.10
           is_reserved_st            44            8.10
    is_data_quality_issue             2            0.37


## 8. Persisting Cleaned Datasets to `data/processed/`
Save the validated dataset to CSV, high-performance Parquet, and JSON metadata.


In [9]:
processed_dir = os.path.join("..", "data", "processed")
if not os.path.exists(processed_dir):
    processed_dir = os.path.join("data", "processed")

os.makedirs(processed_dir, exist_ok=True)

csv_out = os.path.join(processed_dir, "allocated_limit_mps_cleaned.csv")
parquet_out = os.path.join(processed_dir, "allocated_limit_mps_cleaned.parquet")
json_out = os.path.join(processed_dir, "summary_totals_metadata.json")

# 1. Save CSV
operational_df.to_csv(csv_out, index=False, encoding='utf-8')

# 2. Save Parquet
operational_df.to_parquet(parquet_out, index=False)

# 3. Save Summary Metadata
meta_dict = {
    "source_filename": "Allocated Limit for Honble MPs.csv",
    "official_grand_total_inr": official_grand_total,
    "calculated_sum_inr": float(operational_df['allocated_amount_inr'].sum()),
    "total_mp_records": len(operational_df),
    "is_sum_reconciled": bool(np.isclose(operational_df['allocated_amount_inr'].sum(), official_grand_total)),
    "reconciliation_delta": float(round(operational_df['allocated_amount_inr'].sum() - official_grand_total, 2)),
    "records_with_quality_flags": int(operational_df['is_data_quality_issue'].sum())
}

with open(json_out, "w", encoding='utf-8') as f:
    json.dump(meta_dict, f, indent=2)

print("Processed Artifacts Saved Successfully:")
print(f" - CSV: {csv_out} ({os.path.getsize(csv_out):,} bytes)")
print(f" - Parquet: {parquet_out} ({os.path.getsize(parquet_out):,} bytes)")
print(f" - Metadata: {json_out} ({os.path.getsize(json_out):,} bytes)")


Processed Artifacts Saved Successfully:
 - CSV: data\processed\allocated_limit_mps_cleaned.csv (56,286 bytes)
 - Parquet: data\processed\allocated_limit_mps_cleaned.parquet (34,377 bytes)
 - Metadata: data\processed\summary_totals_metadata.json (280 bytes)


## 9. Final Verification of Cleaned Dataset
Re-load the processed parquet dataset and display a preview of the standardized records.


In [10]:
verification_df = pd.read_parquet(parquet_out)
print(f"Reloaded Cleaned Dataset Shape: {verification_df.shape[0]} rows × {verification_df.shape[1]} columns")
print("\nFirst 5 Records:")
print(verification_df[['sr_no', 'state', 'mp_name', 'constituency_clean', 'allocated_amount_inr', 'is_reserved_sc', 'is_reserved_st']].head(5).to_string())


Reloaded Cleaned Dataset Shape: 543 rows × 12 columns

First 5 Records:
   sr_no              state                         mp_name constituency_clean  allocated_amount_inr  is_reserved_sc  is_reserved_st
0      1        Maharashtra  AASHTIKAR PATIL NAGESH BAPURAO            HINGOLI          190289442.00           False           False
1      2  Jammu And Kashmir             ABDUL RASHID SHEIKH         BARAMULLAH          154773472.11           False           False
2      3              Bihar               ABHAY KUMAR SINHA         AURANGABAD          147000000.00           False           False
3      4        West Bengal            ABHIJIT GANGOPADHYAY             TAMLUK          147000000.00           False           False
4      5        West Bengal                  Abu Taher Khan        MURSHIDABAD          147000000.00           False           False
